In [3]:
from openai import AsyncOpenAI
import pandas as pd
import ast
import re
from difflib import SequenceMatcher

with open("/Users/sagewong/git/StigmatizingLanguageProject/ChatGPTAPIKey.txt") as file:
    chatGPTAPIKey = file.read()
client = AsyncOpenAI(api_key=chatGPTAPIKey)

def cleanOllamaOutput(output):
    pattern = r"\[.*?\]"
    
    matches = re.findall(pattern, output, re.DOTALL)
    a = matches[0].replace("\n", "")
    escaped_string = re.sub(r"(?<=\w)'(?=\w)", r"\'", a)
    result = re.sub(r"\([^()]*\)", "", escaped_string)
    return ast.literal_eval(result.replace("\\n", "").replace("\\\\", "\\"))

annotatedDataset = pd.read_csv("/Users/sagewong/git/StigmatizingLanguageProject/FinalFinalAnnnotatedData.csv")


In [4]:
true_positive = 0
false_positive = 0
true_negative = 0
false_negative = 0
for index in range(annotatedDataset.shape[0]):
    print("INDEX: " + str(index))
    clinicalNote = annotatedDataset.iloc[index]['Completion']

    answer = ast.literal_eval(annotatedDataset.iloc[index]['annotated'])


    prompt = "You are a professional linguist researcher who is trying to identify stigmatizing language in clinical notes. Given this clinical note, return to me in a python-type list all forms of stigmatizing language (e.g. noncompliant, nonadherent, challenging, uncooperative, refused, contradicting themselves, frequent visitor to ED, narcotic dependence, obese, alcoholic, inconsistent responses etc...). Do not include any descriptions or explanations. DO NOT INCLUDE STIGMATIZING LANGUAGE IF IT IS NOT FOUND IN THE NOTE, ONLY INCLUDE LANGUAGE THAT IS IN THE NOTE. Also do not rewrite the stigmatizing language in your own words. Here's the actual note you will have to analyze, and make sure you output the list of stigmatizing words in JSON output: " + clinicalNote

    completion = await client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}])
    response = completion.choices[0].message.content
    rawOutput = response
    cleanedOutput = cleanOllamaOutput(rawOutput)
    print(cleanedOutput)
    print(answer)

    while True:
        try:
            prompt = "Return to me only a list of elements which are referring to the same thing in these two lists in JSON format. Do not include explanations to the text, only include the text itself. Here are the two lists: " + str(cleanedOutput) + ", " + str(answer)
            completion = await client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}])
            response = completion.choices[0].message.content
            matching = cleanOllamaOutput(response)
            break
        except:
            print()

    print(matching)

    true_positive += len(matching)
    false_positive += len([i for i in cleanedOutput if i not in matching])
    true_negative += clinicalNote.count(" ")/3 - len(matching) + len([i for i in answer if answer not in matching])*3
    false_negative += len([i for i in answer if answer not in matching])

    tempTruePositive = len(matching)
    tempFalsePositive = len([i for i in cleanedOutput if i not in matching])
    tempTrueNegative = clinicalNote.count(" ")/3 - len(matching) + len([i for i in answer if answer not in matching])*3
    tempFalseNegative = len([i for i in answer if answer not in matching])
    tempPrecision = tempTruePositive/(tempTruePositive + tempFalsePositive)
    tempRecall = tempTruePositive/(tempTruePositive + tempFalseNegative)
    try:    
        tempf1score = (2*tempPrecision*tempRecall)/(tempPrecision+tempRecall)
        print("SINGLE NOTE F1 SCORE: " + str(tempf1score))
    except:
        print("SINGLE NOTE F1 SCORE: 0")
    precision = true_positive/(true_positive + false_positive)
    recall = true_positive/(true_positive + false_negative)

    f1_score = (2*precision*recall)/(precision+recall)
    print("TOTAL F1 SCORE: " + str(f1_score))

precision = true_positive/(true_positive + false_positive)
recall = true_positive/(true_positive + false_negative)

f1_score = (2*precision*recall)/(precision+recall)
print(precision)
print(recall)
print(f1_score)

INDEX: 0
['challenging', 'uncooperative', 'noncompliant', 'frequent cursing at nurses', 'difficult patient', 'challenging interactions', 'unreliable historical recounting', 'refused', 'resistant to modifying his diet', 'cursing at staff', 'noncompliance', 'significant barrier to his care']
['challenging patient', 'uncooperative behavior', 'noncompliant', 'citing his preferences for junk food', 'frequent cursing at nurses', 'difficult patient', 'challenging interactions', 'unreliable historical recounting', 'refused several exams', 'resistant to modifying his diet', 'complicating our attempts to provide care', 'significant barrier to his care', 'noncompliance has been a significant barrier to his care', "Mohammad's noncompliance has been a significant barrier to his care"]
['challenging', 'uncooperative', 'noncompliant', 'frequent cursing at nurses', 'difficult patient', 'challenging interactions', 'unreliable historical recounting', 'refused', 'resistant to modifying his diet', 'noncom

CancelledError: 